<a href="https://colab.research.google.com/github/Haruki-24/Challenge-Alura-Agente-OCI/blob/main/Challenge_Alura_Agente_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto: Agente de Operations & Control Industrial (OCI)

## 0 - Configuracion de entorno

### Instalación de depedencias

In [ ]:
# 1. Creamos el archivo requirements.txt incluyendo el paquete faltante
requirements_content = """
faiss-cpu
google-generativeai
langchain
langchain-community
langchain-core
langchain-google-genai
langgraph
openpyxl
pandas
PyMuPDF
pypdf
python-dotenv
unstructured
langchain-experimental
langchain-huggingface
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

print("✅ Archivo 'requirements.txt' actualizado con langchain-huggingface.")

# 2. Instalamos de manera limpia
!pip install -q -r requirements.txt
print("🚀 ¡Entorno configurado con éxito!")

✅ Archivo 'requirements.txt' actualizado con langchain-huggingface.
🚀 ¡Entorno configurado con éxito!


### Importación de biibliotecas y conexión con LLMs

In [ ]:
import os
from getpass import getpass
import pandas as pd
# Importaciones clave de LangChain / Google GenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

print("✅ Bibliotecas base importadas con éxito.")

✅ Bibliotecas base importadas con éxito.


/tmp/ipykernel_19395/4043027227.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
#Recuperacion y validacion de Api key
from google.colab import userdata
GEMINI_API_KEY = userdata.get('Gemini-GC')

## 1 - Extracción y carga de datos

In [ ]:

!pip install -q "unstructured[pdf]" # 1. Asegurar la instalación de las dependencias necesarias

!pip show unstructured # 2. Verificación de la instalación

Name: unstructured
Version: 0.23.1
Summary: A library that prepares raw documents for downstream ML tasks.
Home-page: https://github.com/Unstructured-IO/unstructured
Author: 
Author-email: Unstructured Technologies <devops@unstructuredai.io>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: beautifulsoup4, charset-normalizer, emoji, filelock, filetype, html5lib, installer, langdetect, lxml, numba, numpy, psutil, python-iso639, python-magic, python-oxmsg, rapidfuzz, regex, requests, spacy, tqdm, typing-extensions, unstructured-client, wrapt
Required-by: 


In [ ]:
# Importaciones necesarias para el cargador fusionado
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, UnstructuredExcelLoader, TextLoader
from langchain_community.document_loaders.merge import MergedDataLoader

# Carga de los documentos desde tu ruta especificada
url_docs = '/content/drive/MyDrive/Colab Notebooks/Challenge Agente OCI/docs'

# 3. Definición de cargadores por tipo de archivo
# Los PDFs usan PyPDFLoader
loader_pdfs = DirectoryLoader(url_docs, glob="*.pdf", loader_cls=PyPDFLoader)

# Los Excel usan UnstructuredExcelLoader
loader_excel = DirectoryLoader(url_docs, glob="*.xlsx", loader_cls=UnstructuredExcelLoader)

# 4. Fusión de cargadores
all_loaders = MergedDataLoader(loaders=[loader_pdfs, loader_excel])

# 5. Carga final
print("⏳ Cargando documentos...")
documentos_oci = all_loaders.load()

# 6. Validación
print(f"✅ Se han cargado exitosamente {len(documentos_oci)} documentos en total.")

if len(documentos_oci) > 0:
    print(f"📄 Primer documento cargado: {documentos_oci[0].metadata.get('source', 'Desconocido')}")

⏳ Cargando documentos...
✅ Se han cargado exitosamente 7 documentos en total.
📄 Primer documento cargado: /content/drive/MyDrive/Colab Notebooks/Challenge Agente OCI/docs/Politica_HSE_OCI.pdf


In [ ]:
documentos_oci


[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-27T02:42:36+00:00', 'title': 'Desafío de Agente IA para Documentos - Google Gemini', 'moddate': '2026-06-27T02:42:36+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/Challenge Agente OCI/docs/Politica_HSE_OCI.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Política de Salud, Seguridad y Medio Ambiente (HSE) - OCI\n1. Compromiso Fundamental\nEn Operations & Control Industrial (OCI), reconocemos que la integridad de nuestro personal, la protección dela salud y el cuidado del medio ambiente son valores innegociables. Ninguna tarea operativa es tan urgente ni tanprioritaria que no pueda realizarse de manera segura y responsable. Nuestro compromiso es entregar solucionestecnológicas de vanguardia sin comprometer nunca nuestros estándares HSE.\n2. Pilares de Nuestra

### Extracción PDF

### Extracción csv y Excel

## 2 - Fragmentación

In [ ]:
# Splitter de texto para PDFs (ej. RecursiveCharacterTextSplitter)
# Estrategia de parseo para filas de CSV/Excel (guardando ID_Pozo como metadato)

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuramos el splitter para documentos técnicos
# chunk_size=500: Un buen tamaño para que el LLM tenga contexto sin perderse
# chunk_overlap=50: Mantiene una pequeña continuidad entre bloques
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Ahora dividimos los documentos
documentos_fragmentados = text_splitter.split_documents(documentos_oci)

# Verificamos
print(f"✅ Documentos originales: {len(documentos_oci)}")
print(f"✅ Fragmentos (chunks) resultantes: {len(documentos_fragmentados)}")

# Inspección rápida del primer fragmento
print("\n--- Ejemplo de fragmento ---")
print(documentos_fragmentados[0].page_content)

✅ Documentos originales: 7
✅ Fragmentos (chunks) resultantes: 34

--- Ejemplo de fragmento ---
Política de Salud, Seguridad y Medio Ambiente (HSE) - OCI
1. Compromiso Fundamental
En Operations & Control Industrial (OCI), reconocemos que la integridad de nuestro personal, la protección dela salud y el cuidado del medio ambiente son valores innegociables. Ninguna tarea operativa es tan urgente ni tanprioritaria que no pueda realizarse de manera segura y responsable. Nuestro compromiso es entregar solucionestecnológicas de vanguardia sin comprometer nunca nuestros estándares HSE.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Usamos el encoder de tiktoken para contar tokens reales
# Asegúrate de tener instalado: pip install tiktoken
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,    # Ahora esto son 500 TOKENS, no caracteres
    chunk_overlap=50   # 50 tokens de solapamiento
)

# Fragmentamos
documentos_fragmentados = text_splitter.split_documents(documentos_oci)

# Verificación
print(f"✅ Fragmentos generados: {len(documentos_fragmentados)}")

✅ Fragmentos generados: 12


## 3 - Vectorización (Embedings)

In [ ]:
# Inicialización del modelo de Embeddings
# Creación e indexación en base de datos vectorial local (ej. FAISS o Chroma)

from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

# 1. Cargamos el tokenizer del modelo
model_id = 'intfloat/multilingual-e5-small'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Configuramos el splitter para que respete el límite de tokens (512)
# Usamos un chunk_size menor (ej. 450) para dejar margen al overlap y evitar el error
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=450,
    chunk_overlap=50
)

# 3. Fragmentamos
hf_fragmentos = text_splitter.split_documents(documentos_oci)

print(f"✅ Fragmentación completada.")
print(f"Total de fragmentos generados: {len(hf_fragmentos)}")

✅ Fragmentación completada.
Total de fragmentos generados: 11


In [ ]:
hf_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=512,
    chunk_overlap=50
)

In [ ]:
hf_fragmentos = hf_splitter.split_documents(documentos_oci)
len(hf_fragmentos)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (546 > 512). Running this sequence through the model will result in indexing errors


7

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Cargamos el modelo local (eficiente para fragmentar sin llamadas a API)
hf_emb_model = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small')

# 2. Configuramos el SemanticChunker
# breakpoint_threshold_type: "percentile" suele ser el más estable para textos técnicos
semantic_hf_splitter = SemanticChunker(
    hf_emb_model,
    breakpoint_threshold_type="percentile"
)

# 3. Fragmentación semántica
print("⏳ Fragmentando semánticamente (esto puede tomar un momento)...")
semantic_hf_chunks = semantic_hf_splitter.split_documents(documentos_oci)

print(f"✅ Fragmentación semántica completada: {len(semantic_hf_chunks)} fragmentos.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

⏳ Fragmentando semánticamente (esto puede tomar un momento)...
✅ Fragmentación semántica completada: 15 fragmentos.


## 4 -Recuperacion RAG

### 1. Transformación de la pregunta en embedding

In [ ]:
# Función de búsqueda semántica
# Implementación de filtros por metadatos (ej. filtrar solo por "Pozo PCP")
# Componente de Reranking (opcional, para mejorar precisión)

###  2. Búsqueda semántica en la base de datos vectorial

### 3. Filtrado por metadados

### 4. Reclasificación (Reranking)

### 5. Ensamblaje del contexto

## 5 - Producción y validación de respuestas

In [ ]:
# Prompt Engineering (Definiendo el rol del Agente OCI y la regla "Human-in-the-loop")
# Lógica de fallback: "No cuento con esa información en los manuales actuales..."
# Función final de la aplicación

### 1. Generación de la respuesta

### 2. Citación de la fuente

### 3. Validación y control de alucinación

### 4. Fallback (alternativa) cuando no hay respuesta

### 5. Formato final